# S7c — prepare the server folder and try the web API inside Colab (CPU is enough)

specs/s7c-serve/spec.md (approved 26 Nov 2026). Makes the private folder a server needs, starts the API here
for a few minutes, sends test requests and times them. Nothing is deployed to the internet from this notebook.


## 1. Drive and data

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
DRIVE = "/content/drive/MyDrive/anthro"
DRIVE_ARCHIVE = "/content/drive/MyDrive/datasets/bodym.tar"
DATA_DIR = "/content/bodym"
os.makedirs(DRIVE, exist_ok=True)

if os.path.exists(DATA_DIR):
    print("Data already on this runtime.")
elif os.path.exists(DRIVE_ARCHIVE):
    !tar -xf "{DRIVE_ARCHIVE}" -C /content
else:
    !pip -q install awscli
    !aws s3 sync --no-sign-request s3://amazon-bodym/ {DATA_DIR} --only-show-errors
    os.makedirs(os.path.dirname(DRIVE_ARCHIVE), exist_ok=True)
    !tar -cf "{DRIVE_ARCHIVE}" -C /content bodym

## 2. Get the code (with the API packages)

In [ ]:
BRANCH = "s7c-serve"   # set to "main" once merged
REPO = "https://github.com/Anthropometry/Anthropometry_Accademic_Model.git"
!rm -rf /content/repo && git clone -q --branch {BRANCH} {REPO} /content/repo
!pip -q install -e "/content/repo[serve]"
!git -C /content/repo log -1 --format="commit %h  %s"
GEO, CACHE = f"{DRIVE}/geo", f"{DRIVE}/cache"
RUN = f"{DRIVE}/runs/r4"
ANSUR = f"{DRIVE}/ansur2"
FORMULAS = f"{ANSUR}/ansur_formulas.json"

# A package installed in a running Colab session is only importable after a restart;
# pointing Python at the code directly avoids that.
import sys
sys.path.insert(0, "/content/repo/src")
import os
os.environ["TORCH_HOME"] = f"{DRIVE}/torch_cache"   # keep the downloaded tool weights between sessions
PHOTOS = f"{DRIVE}/test_photos"


## 3. Make the server folder (seconds)
Writes `MyDrive/anthro/serving/` with `model.pt`, `eval_report.json`, `MANIFEST.json`. Keep it private.
**Paste back:** the printed manifest.


In [ ]:
SERVING = f"{DRIVE}/serving"
!anthro prepare-serving --ckpt {RUN}/best.pt --data {DATA_DIR} --out-dir {SERVING}


## 4. Start the API here, in the background
The key below is only for this Colab test; a real server gets its own long, secret key.


In [ ]:
import os
import subprocess
import sys
import time

import requests

env = dict(os.environ, ANTHRO_MODEL_DIR=SERVING, ANTHRO_API_KEY="colab-test-only", PYTHONPATH="/content/repo/src")
server = subprocess.Popen([sys.executable, "-m", "uvicorn", "anthro.serve:create_app_from_env", "--factory", "--port", "8080"],
                          env=env, stdout=open("/content/api.log", "w"), stderr=subprocess.STDOUT)
for _ in range(120):
    try:
        r = requests.get("http://localhost:8080/v1/health", timeout=2)
        if r.status_code in (200, 503):
            break
    except requests.ConnectionError:
        pass
    time.sleep(2)
print(r.status_code, r.json())


## 5. Five requests with your first test photo pair (from notebook 07), timed
**Paste back:** the five times and the status codes.


In [ ]:
import pandas as pd
p = pd.read_csv(f"{PHOTOS}/pairs.csv").iloc[0]
times = []
for i in range(5):
    t0 = time.perf_counter()
    with open(p.front, "rb") as f, open(p.side, "rb") as s:
        r = requests.post("http://localhost:8080/v1/measure", headers={"X-API-Key": "colab-test-only"},
                          files={"front": f, "side": s},
                          data={"height_cm": p.height_cm, "sex": p.sex, "age_range": p.get("age_range", "25-34")},
                          timeout=300)
    times.append(round(time.perf_counter() - t0, 1))
    print(i + 1, r.status_code, f"{times[-1]} s", r.json().get("error", ""))
print("seconds:", times, "| first request includes warming up")


## 6. Stop the API

In [ ]:
server.terminate()
print(open("/content/api.log").read()[-800:])   # status and timing lines only; no photos or values


## Next
Download the three files in `MyDrive/anthro/serving/` and put them in the `serving/` folder of the repository
on the machine that builds the Docker image (your laptop first). Follow `docs/DEPLOY.md`.

**Before committing this notebook:** Edit → Clear all outputs, then save (POLICY 6).
